# Scoring a recording with nyx

This notebook scores one recording into WAKE / NREM / REM.

It replaces `run_pipeline_new.ipynb`. The analysis is identical — the difference is
that the logic now lives in `nyx.pipeline`, so each cell is a few lines and the
points where *you* have to make a decision are the only places you edit.

**There are three such decisions**, marked 🔧 below:

1. the analysis window (trim artefacts at the start/end),
2. the EMG wake/sleep threshold,
3. which cluster is REM and which is NREM.

Each one has an automatic default. Run the notebook top to bottom first, look at the
plots, and only override what looks wrong.

A manually scored reference is **optional** — skip section 6 if you don't have one.

## 1. Setup

In [ ]:
import os

# Keep the linear-algebra libraries to one thread each: the pipeline is not
# parallel, and oversubscribing slows it down considerably.
for _var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
             "NUMEXPR_NUM_THREADS", "BLIS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_var] = "1"

import matplotlib.pyplot as plt
import numpy as np

import nyx
from nyx import plotting

## 2. Load the recording

Point `nyx.read_recording` at your file. `eeg_channel` / `emg_channel` are either
positions (`0` = first channel in the file) or channel names.

Always check `describe()` before going further — picking the wrong channel is the
single most common mistake, and everything downstream will look plausible but be wrong.

In [ ]:
RECORDING_PATH = r"...\A31_D04.edf"   # <- your file
PARAMS_PATH = "../params/mice.json"
OUTPUT_DIR = "../results/A31_D04"

recording = nyx.read_recording(RECORDING_PATH, eeg_channel=0, emg_channel=1)
params = nyx.load_params(PARAMS_PATH)

print(recording.describe())

### Datasets nyx already knows

If you downloaded one of the public datasets from the paper, the recipe knows the
folder layout and the scoring format, so you can skip the cell above:

```python
from nyx.datasets import list_datasets, load_dataset

list_datasets()
recording, reference = load_dataset(
    "gulledge-2025",
    data_root=r"...\data",
    sub_dataset="Saline (N=4)",
    recording_name="A31_D04",
)
```

## 3. 🔧 Decision 1 — the analysis window

Look at the start and end of the EEG. If there are artefacts (flat line, saturation,
the animal being handled), narrow the window.

In [ ]:
import spikeinterface.widgets as sw

sw.plot_traces(recording.eeg, time_range=[0, min(4000, recording.duration)], backend="matplotlib")
sw.plot_traces(recording.eeg, time_range=[max(0, recording.duration - 4000), recording.duration],
               backend="matplotlib")

In [ ]:
# 🔧 None = use the whole recording. Otherwise (start, end) in seconds.
WINDOW = None

windowed = recording if WINDOW is None else recording.time_slice(*WINDOW)
window = (0.0, recording.duration) if WINDOW is None else WINDOW
print(f"Analysing {window[0]:g}-{window[1]:g}s of {recording.duration:.0f}s")

## 4. 🔧 Decision 2 — the EMG wake/sleep threshold

The threshold should sit in the valley between the two modes of the histogram.

In [ ]:
emg = nyx.compute_emg_features(windowed, params)

# Epochs below NOSIGNAL are treated as signal loss, not as sleep. 0.0 disables it.
NOSIGNAL = 0.0

auto_threshold = nyx.find_wake_sleep_threshold(emg, nosignal_threshold=NOSIGNAL)
print(f"Automatic EMG threshold: {auto_threshold:.3f}")

plt.figure(figsize=(7, 4))
plt.hist(emg.power[emg.power > NOSIGNAL], bins=100, density=True, alpha=0.5)
plt.axvline(auto_threshold, color="green", ls="--", label=f"auto = {auto_threshold:.3f}")
if NOSIGNAL > 0:
    plt.axvline(NOSIGNAL, color="grey", ls=":", label=f"nosignal = {NOSIGNAL:.3f}")
plt.xlabel("EMG power"); plt.ylabel("density"); plt.legend(); plt.show()

In [ ]:
# 🔧 None = keep the automatic threshold. Set a number to override it.
EMG_THRESHOLD = None

wake_sleep = nyx.classify_wake_sleep(
    emg,
    threshold=EMG_THRESHOLD,
    nosignal_threshold=NOSIGNAL,
    min_duration=params.get("scoring", {}).get("min_duration", 4.0),
)
print(f"Using {wake_sleep.threshold:.3f} ({wake_sleep.threshold_source})")

plotting.plot_hypnogram(
    __import__("pandas").DataFrame(wake_sleep.hypnogram),
    possible_labels=["WAKE", "SLEEP", "NOSIGNAL"],
    title="Wake / sleep",
)
plt.show()

## 5. EEG PCA and clustering

PCA is fitted on the sleep epochs only, then those epochs are clustered in PC space.

In [ ]:
pca = nyx.compute_sleep_pca(windowed, params, wake_sleep)
print(f"{pca.scores.shape[0]} sleep epochs, {pca.scores.shape[1]} components")

fig, axs = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for i, ax in enumerate(axs.ravel()[: pca.pca.n_components_]):
    ax.plot(pca.freqs, pca.pca.components_[i])
    ax.set_title(f"PC{i + 1}: {100 * pca.explained_variance_ratio[i]:.1f}% var")
    ax.set_xlabel("Frequency (Hz)")
plt.tight_layout(); plt.show()

In [ ]:
# Defaults come from nyx.pipeline.DEFAULT_CLUSTERING, overridden by the
# "clustering" section of the params file. Override here to experiment.
clustering = {
    "method": "hdbscan",           # hdbscan | kmeans | gmm | dbscan
    "n_clusters": 2,               # kmeans / gmm only
    "pcs_to_use": [0, 1, 2, 3],
    "use_emg": False,
    "outlier_z_threshold": 6,
    "hdbscan_min_cluster_size": 300,
    "hdbscan_min_samples": 30,
}

clusters = nyx.cluster_sleep(pca, wake_sleep, clustering=clustering, emg=emg)
print("cluster sizes:", clusters.sizes())

In [ ]:
# Clusters in PC space, and the mean PSD of each — this is what tells you which
# cluster is REM: REM has less low-frequency (delta) power than NREM.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
cmap = plt.get_cmap("Set1")

for i, cid in enumerate(clusters.unique_labels):
    m = clusters.labels == cid
    ax1.scatter(clusters.features_scaled[m, 0], clusters.features_scaled[m, 1],
                s=6, color=cmap(i), alpha=0.5, label=f"C{cid}")
for i, c in enumerate(clusters.centers):
    ax1.scatter(c[0], c[1], c="k", marker="x", s=120, lw=2)
    ax1.annotate(f"C{i}", (c[0], c[1]), xytext=(6, 4), textcoords="offset points", fontweight="bold")
ax1.set_xlabel(f"PC{clusters.pcs_to_use[0] + 1}")
ax1.set_ylabel(f"PC{clusters.pcs_to_use[1] + 1}")
ax1.set_title("Clusters in PC space"); ax1.legend(markerscale=3)

sleep_bins = ~np.any(np.isnan(pca.signal), axis=1)
spec = pca.spectrogram[:, sleep_bins][:, clusters.valid_mask]
for i, cid in enumerate(clusters.unique_labels):
    m = clusters.labels == cid
    mean = spec[:, m].mean(axis=1)
    ci = 1.96 * spec[:, m].std(axis=1) / np.sqrt(m.sum())
    ax2.plot(pca.freqs, mean, color=cmap(i), label=f"C{cid} (n={m.sum()})")
    ax2.fill_between(pca.freqs, mean - ci, mean + ci, color=cmap(i), alpha=0.25)
ax2.set_xlabel("Frequency (Hz)"); ax2.set_ylabel("Power (dB)")
ax2.set_title("Mean PSD per cluster"); ax2.legend()
plt.tight_layout(); plt.show()

## 🔧 Decision 3 — which cluster is REM

Clusters are ordered by their centroid on the first PC used, and `STAGE_ORDER` names
them lowest → highest. Check against the PSD plot above; if it is the wrong way
round, either flip `STAGE_ORDER` or set `OVERRIDES`.

In [ ]:
STAGE_ORDER = ["REM", "NREM"]     # 🔧 low PC1 -> high PC1
OVERRIDES = None                   # 🔧 e.g. {0: "NREM", 1: "REM"}

staging = nyx.assign_stages(clusters, pca, wake_sleep,
                            stage_order=STAGE_ORDER, overrides=OVERRIDES)

for cid, stage in staging.cluster_to_stage.items():
    print(f"  cluster {cid}  PC1 centroid {clusters.centers[cid, 0]:+.3f}  ->  {stage}")

total = sum(staging.stage_durations().values())
for stage, seconds in sorted(staging.stage_durations().items()):
    print(f"  {stage:<10} {seconds:9.1f}s ({100 * seconds / total:.1f}%)")

In [ ]:
plotting.plot_hypnogram(
    __import__("pandas").DataFrame(staging.hypnogram),
    possible_labels=["WAKE", "REM", "NREM", "NOSIGNAL"],
    title="Final hypnogram",
)
plt.show()

### Refining a cluster by hand (optional)

Draw a polygon around the epochs you want to relabel, then rebuild the hypnogram
from the mask:

```python
%matplotlib qt                     # or %matplotlib widget over SSH
from nyx.interactive import PolygonSelector

fig, ax = plt.subplots(figsize=(12, 9))
ax.scatter(clusters.features_scaled[:, 0], clusters.features_scaled[:, 1], s=5)
selector = PolygonSelector(ax, clusters.features_scaled[:, :2])
plt.show()
```

After drawing the polygon and pressing `enter`:

```python
import numpy as np
from nyx.clustering import reconstruct_signal_multiclass
from nyx.scoring import prepare_epoch_data

rem_mask = selector.get_mask()          # True = REM

# -2 NOSIGNAL, -1 WAKE, 0 NREM, 1 REM
stages = np.full(len(clusters.valid_mask), -2, dtype=int)
stages[clusters.valid_mask] = rem_mask.astype(int)

signal = reconstruct_signal_multiclass(stages, wake_sleep.hypnogram,
                                       pca.fs, pca.signal.shape[0])
manual_hypnogram = prepare_epoch_data(
    signal, pca.fs,
    state_names={-2: 'NOSIGNAL', -1: 'WAKE', 0: 'NREM', 1: 'REM'},
)
```

## 6. Compare with a manual scoring — OPTIONAL

Only if you have one. nyx does not need it.

`read_annotations` handles the common layouts; tell it how your file is organised:

| your file | call |
|---|---|
| `time,duration,label` columns | `read_annotations(path)` |
| one row per epoch, stage as a number | `read_annotations(path, format="epoch_csv", epoch_length=10, label_map={1: "WAKE", 2: "NREM", 3: "REM"})` |
| one column per animal | `read_annotations(path, format="column_csv", column="A31", epoch_length=4, label_map={...})` |
| visbrain / somnotate `.hyp` | `read_annotations(path)` |

In [ ]:
REFERENCE_PATH = None   # e.g. r"...\A31_D04_staging.csv"

agreement = None
if REFERENCE_PATH:
    reference = nyx.read_annotations(
        REFERENCE_PATH,
        format="epoch_csv",
        stage_column="Rodent Sleep",
        time_column="Time Stamp",
        time_format="clock",
        label_map={3: "NREM", 4: "REM", 5: "WAKE"},
    )
    agreement = nyx.evaluate(staging, reference, window=window)
else:
    print("No reference scoring — skipping. This is fine.")

## 7. Save

In [ ]:
from nyx.pipeline import ScoringResult

result = ScoringResult(
    recording=windowed, emg=emg, wake_sleep=wake_sleep, pca=pca,
    clusters=clusters, staging=staging, agreement=agreement,
    window=window, total_duration=recording.duration, params=params,
)

nyx.save_results(result, OUTPUT_DIR)
print(f"Saved to {os.path.abspath(OUTPUT_DIR)}")

---
## All of the above, in one call

When the defaults are good enough — for a batch, or once you have checked one
recording of a set — the whole notebook collapses to:

```python
result = nyx.score_recording(recording, params, window=WINDOW, reference=reference)
nyx.save_results(result, OUTPUT_DIR)
```